In [1]:
 """SESSION 24 — genuine CondInst R101-DCNv2 with five-fold OOF CV.
"""

from __future__ import annotations

# %% 1. Imports and shared, already-audited CV utilities
import argparse
import csv
import hashlib
import json
import multiprocessing
import os
import random
import shutil
import statistics
import subprocess
import sys
import time
import zipfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import urlretrieve

# Session 24 is a fully self-contained script.
# No cross-session imports — SESSION_23 is not importable in a separate Kaggle cell.


os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ["MPLBACKEND"] = "Agg"

SESSION_NAME = "session24_condinst_r101_dcn_cv_skindisnet2"
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
MMCV_INDEX = "https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html"
TORCH_INDEX = "https://download.pytorch.org/whl/cu121"
PY312_ENV = Path("/kaggle/working/session24_condinst_py312")
CONDINST_R50_URL = (
    "https://download.openmmlab.com/mmdetection/v3.0/condinst/"
    "condinst_r50_fpn_ms-poly-90k_coco_instance/"
    "condinst_r50_fpn_ms-poly-90k_coco_instance_20221129_125223-4c186406.pth"
)


def _pip(*args: str) -> None:
    print(f"  [install] {' '.join(args)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


def _mmdet_tool(name: str) -> str:
    import mmdet
    root = Path(mmdet.__file__).resolve().parent
    for candidate in [
        root / ".mim" / "tools" / f"{name}.py",
        root / "tools" / f"{name}.py",
        root.parent / "tools" / f"{name}.py",
    ]:
        if candidate.is_file():
            return str(candidate)
    raise FileNotFoundError(f"MMDetection tools/{name}.py not found under {root}")


def _parse_folds(value: str) -> list[int]:
    folds = sorted({int(part.strip()) for part in value.split(",") if part.strip()})
    if not folds or any(fold not in range(5) for fold in folds):
        raise ValueError("--folds must contain comma-separated values from 0 through 4")
    return folds


def _relaunch_python312(args: argparse.Namespace) -> int | None:
    """Relaunch under the Python version supported by the pinned MMCV wheel."""
    if sys.version_info[:2] == (3, 12):
        return None
    work = Path("/kaggle/working")
    if not work.is_dir():
        raise RuntimeError(
            "This OpenMMLab setup requires Python 3.12. On Kaggle, run this "
            "script from a notebook with /kaggle/working available."
        )

    uv = [sys.executable, "-m", "uv"]
    try:
        subprocess.run(
            [sys.executable, "-m", "uv", "--version"],
            check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
    except (subprocess.CalledProcessError, FileNotFoundError):
        _pip("uv")

    py = PY312_ENV / "bin" / "python"
    if not py.is_file():
        subprocess.run([*uv, "python", "install", "3.12"], check=True)
        subprocess.run(
            [*uv, "venv", "--seed", "--python", "3.12", str(PY312_ENV)],
            check=True,
        )
    version = subprocess.check_output(
        [str(py), "-c", "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')"],
        text=True,
    ).strip()
    if version != "3.12":
        raise RuntimeError(f"Python bootstrap created {version}, expected 3.12")

    runtime = work / "SESSION_24_runtime.py"
    source_path = globals().get("__file__")
    if source_path:
        source = Path(source_path).expanduser().resolve()
        if not source.is_file():
            raise FileNotFoundError(f"Session 24 source file does not exist: {source}")
        if source != runtime.resolve():
            shutil.copy2(source, runtime)
    else:
        try:
            history = get_ipython().user_ns.get("In", [])  # type: ignore[name-defined]
        except NameError as error:
            raise RuntimeError(
                "Cannot locate Session 24 source: __file__ is unavailable and "
                "the current process has no IPython input history."
            ) from error
        source_text = next(
            (
                text for text in reversed(history)
                if isinstance(text, str)
                and "def _relaunch_python312(" in text
                and "def main(" in text
                and "SESSION_24" in text
            ),
            None,
        )
        if source_text is None:
            raise RuntimeError(
                "Cannot recover the complete Session 24 source from notebook "
                "history. Save/upload the script as a .py file and rerun."
            )
        compile(source_text, str(runtime), "exec")
        runtime.write_text(source_text, encoding="utf-8")
    cli = [
        "--data-root", args.data_root,
        "--device", args.device,
        "--folds", args.folds,
        "--max-iters", str(args.max_iters),
        "--image-size", str(args.image_size),
        "--accumulation", str(args.accumulation),
        "--seed", str(args.seed),
    ]
    if args.work_dir:
        cli.extend(["--work-dir", str(args.work_dir)])
    if args.quick:
        cli.append("--quick")
    if not args.skip_existing:
        cli.append("--no-skip-existing")
    if args.resume:
        cli.append("--resume")
    if args.skip_train:
        cli.append("--skip-train")
    print(f"  [python] restarting under Python 3.12: {py}", flush=True)
    return subprocess.call(
        [str(py), "-u", str(runtime), *cli],
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )


def _run_training(config: Path, work_dir: Path, gpu_ids: list[str], resume: bool) -> None:
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = ",".join(gpu_ids)
    train_tool = _mmdet_tool("train")
    if len(gpu_ids) > 1:
        cmd = [
            sys.executable, "-m", "torch.distributed.run",
            "--nproc_per_node", str(len(gpu_ids)),
            "--master_port", str(random.randint(20000, 29999)),
            train_tool, str(config), "--work-dir", str(work_dir),
            "--launcher", "pytorch",
        ]
    else:
        cmd = [sys.executable, train_tool, str(config), "--work-dir", str(work_dir)]
    if resume:
        cmd.append("--resume")
    print(f"  [train] GPUs={gpu_ids}; {' '.join(cmd)}")
    subprocess.check_call(cmd, env=env)


def _run_prediction(config: Path, checkpoint: Path, work_dir: Path, gpu_id: str) -> None:
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = gpu_id
    subprocess.check_call(
        [sys.executable, _mmdet_tool("test"), str(config), str(checkpoint),
         "--work-dir", str(work_dir)],
        env=env,
    )


def _best_checkpoint(work_dir: Path) -> Path | None:
    # MMEngine writes best_coco_segm_mAP_50_iter_*.pth when save_best='coco/segm_mAP_50'
    candidates = sorted(
        work_dir.glob("best_coco_segm_mAP_50_iter_*.pth"),
        key=lambda p: p.stat().st_mtime,
    )
    if candidates:
        return candidates[-1]
    fallback = sorted(work_dir.glob("iter_*.pth"), key=lambda p: p.stat().st_mtime)
    return fallback[-1] if fallback else None


def _polygon_area(xs: list[float], ys: list[float]) -> float:
    return 0.5 * abs(sum(
        xs[i] * ys[(i + 1) % len(xs)] - xs[(i + 1) % len(xs)] * ys[i]
        for i in range(len(xs))
    ))


def _yolo_to_coco(
    items: list[dict], names: list[str], destination: Path, image_ids: dict
) -> dict:
    from PIL import Image
    images, annotations, annotation_id, skipped = [], [], 1, 0
    class_instances: Counter = Counter()
    for item in sorted(items, key=lambda v: image_ids[v["key"]]):
        image_id = image_ids[item["key"]]
        with Image.open(item["image"]) as handle:
            width, height = handle.size
        images.append({"id": image_id, "file_name": str(item["image"]),
                       "width": width, "height": height})
        for line in item["label"].read_text().splitlines():
            values = line.strip().split()
            if not values:
                continue
            if len(values) < 7 or (len(values) - 1) % 2:
                skipped += 1; continue
            try:
                class_id = int(values[0])
                coords = [float(v) for v in values[1:]]
            except ValueError:
                skipped += 1; continue
            if not 0 <= class_id < len(names):
                raise ValueError(f"Invalid class {class_id} in {item['label']}")
            xs = [max(0.0, min(1.0, coords[i])) * width for i in range(0, len(coords), 2)]
            ys = [max(0.0, min(1.0, coords[i])) * height for i in range(1, len(coords), 2)]
            area = _polygon_area(xs, ys)
            x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
            if area <= 1.0 or x1 <= x0 or y1 <= y0:
                skipped += 1; continue
            polygon = [c for pair in zip(xs, ys) for c in pair]
            annotations.append({
                "id": annotation_id, "image_id": image_id,
                "category_id": class_id + 1,
                "segmentation": [polygon], "area": float(area),
                "bbox": [float(x0), float(y0), float(x1 - x0), float(y1 - y0)],
                "iscrowd": 0,
            })
            annotation_id += 1
            class_instances[class_id] += 1
    document = {
        "images": images, "annotations": annotations,
        "categories": [{"id": i + 1, "name": n, "supercategory": "skin"}
                       for i, n in enumerate(names)],
    }
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(document))
    print(f"  [coco] {destination.name}: {len(images)} imgs, "
          f"{len(annotations)} instances, skipped={skipped}")
    return {
        "images": len(images), "annotations": len(annotations),
        "skipped_polygons": skipped,
        "instances_per_class": {names[i]: int(class_instances[i]) for i in range(len(names))},
    }


def _coco_evaluate(ground_truth: Path, predictions: Path, names: list[str]) -> dict:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    detections = json.loads(predictions.read_text())
    if not detections:
        return {"bbox_mAP50": 0.0, "segm_mAP50": 0.0, "segm_mAP50_95": 0.0,
                "per_class_mAP50": {name: 0.0 for name in names}}
    coco_gt = COCO(str(ground_truth))
    coco_dt = coco_gt.loadRes(str(predictions))
    bbox = COCOeval(coco_gt, coco_dt, "bbox")
    bbox.evaluate(); bbox.accumulate(); bbox.summarize()
    segm = COCOeval(coco_gt, coco_dt, "segm")
    segm.evaluate(); segm.accumulate(); segm.summarize()
    per_class = {}
    for cat_id, name in enumerate(names, start=1):
        ev = COCOeval(coco_gt, coco_dt, "segm")
        ev.params.catIds = [cat_id]
        ev.evaluate(); ev.accumulate(); ev.summarize()
        per_class[name] = float(ev.stats[1])
    return {
        "bbox_mAP50": float(bbox.stats[1]),
        "segm_mAP50": float(segm.stats[1]),
        "segm_mAP50_95": float(segm.stats[0]),
        "per_class_mAP50": per_class,
    }


def _merge_oof_predictions(files: list[Path], destination: Path) -> int:
    merged = []
    for path in files:
        if not path.is_file():
            raise FileNotFoundError(f"Missing OOF prediction file: {path}")
        merged.extend(json.loads(path.read_text()))
    destination.write_text(json.dumps(merged))
    return len(merged)


def _balanced_subset(
    items: list[dict], limit: int, seed: int, num_classes: int,
) -> list[dict]:
    if len(items) <= limit:
        return list(items)
    shuffled = list(items)
    random.Random(seed).shuffle(shuffled)
    chosen, used = [], set()
    for class_id in range(num_classes):
        for item in shuffled:
            if item["key"] not in used and class_id in _item_classes(item):
                chosen.append(item); used.add(item["key"]); break
    for item in shuffled:
        if len(chosen) >= limit: break
        if item["key"] not in used:
            chosen.append(item); used.add(item["key"])
    return chosen


def _patch_mmdet_mmcv_guard() -> None:
    """Allow the official Python-3.12 MMCV 2.2 wheel in MMDet 3.3."""
    import importlib.util

    spec = importlib.util.find_spec("mmdet")
    if spec is None or spec.origin is None:
        raise RuntimeError("mmdet was installed but its package path is missing")
    init_path = Path(spec.origin)
    source = init_path.read_text()
    old = "mmcv_maximum_version = '2.2.0'"
    if old in source:
        init_path.write_text(source.replace(
            old, "mmcv_maximum_version = '2.3.0'", 1
        ))
        print(f"  [deps] adjusted MMDetection's MMCV guard: {init_path}")


def _dependency_probe() -> str:
    # Minimal probe: verify packages installed and CUDA ops importable.
    # Does NOT import `from mmdet.models import CondInst` — that triggers the
    # full model registry → seaborn → scipy chain in the subprocess.  The
    # scipy chain fails when disk numpy was left in a damaged state by a prior
    # broken session that ran `pip install numpy==1.26.4`.
    code = (
        "import torch, torchvision, mmcv, mmengine, mmdet; "
        "from mmcv.ops import ModulatedDeformConv2dPack, nms; "
        "assert torch.cuda.is_available(), 'CUDA unavailable'; "
        "print(torch.__version__, torchvision.__version__, "
        "mmcv.__version__, mmengine.__version__, mmdet.__version__)"
    )
    result = subprocess.run(
        [sys.executable, "-c", code],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    if result.returncode != 0:
        raise subprocess.CalledProcessError(
            result.returncode,
            [sys.executable, "-c", "..."],
            output=result.stdout,
        )
    return result.stdout.strip()


def _ensure_dependencies() -> None:
    """Install a deterministic Kaggle Python-3.12 OpenMMLab stack."""
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError(
            "SESSION 24's pinned PyTorch 2.4/MMCV 2.2 CUDA stack requires "
            f"Python 3.12; this interpreter is {sys.version.split()[0]}."
        )
    try:
        print(f"  [deps] usable environment found: {_dependency_probe()}")
        return
    except (subprocess.CalledProcessError, OSError) as error:
        detail = getattr(error, "output", "") or ""
        if detail:
            print("  [deps] current environment probe failed:")
            print("\n".join(detail.strip().splitlines()[-8:]))

    print("  [deps] installing pinned Kaggle packages without openmim ...")
    torch_probe = [
        sys.executable, "-c",
        "import torch; print(torch.__version__.split('+')[0]); "
        "print(torch.version.cuda or 'cpu')",
    ]
    try:
        torch_info = subprocess.check_output(
            torch_probe,
            text=True,
            stderr=subprocess.STDOUT,
        ).splitlines()
    except subprocess.CalledProcessError as error:
        output = error.output or ""
        if "No module named 'torch'" not in output:
            raise RuntimeError(
                "PyTorch is installed but could not be imported in the "
                "Python 3.12 environment:\n" + output[-3000:]
            ) from error
        print(
            "  [deps] isolated Python 3.12 environment has no PyTorch; "
            "installing torch 2.4.0/torchvision 0.19.0 with CUDA 12.1 ...",
            flush=True,
        )
        _pip(
            "--upgrade", "--force-reinstall", "--no-cache-dir",
            "torch==2.4.0", "torchvision==0.19.0",
            "--index-url", TORCH_INDEX,
        )
        torch_info = subprocess.check_output(
            torch_probe, text=True, stderr=subprocess.STDOUT
        ).splitlines()

    torch_version, cuda_version = torch_info[:2]
    if not torch_version.startswith("2.4.") or not cuda_version.startswith("12.1"):
        print(
            f"  [deps] replacing torch {torch_version}/CUDA {cuda_version} "
            "with torch 2.4/cu121"
        )
        _pip(
            "--upgrade", "--force-reinstall", "--no-cache-dir",
            "torch==2.4.0", "torchvision==0.19.0",
            "--index-url", TORCH_INDEX,
        )

    # ── Do NOT downgrade numpy or scipy. ─────────────────────────────────────
    # mmengine AND mmcv declare numpy<2.0 in their PyPI metadata; all OpenMMLab
    # installs must use --no-deps so pip never touches numpy.
    # numpy/scipy are intentionally omitted from the pre-install list; we will
    # repair the disk state below after mmcv/mmdet are in place.
    _pip(
        "--upgrade",
        "addict",                            # mmengine runtime dep
        "termcolor",                         # mmengine runtime dep
        "packaging",                         # mmengine runtime dep
        "opencv-python==4.10.0.84",          # satisfies mmcv / mmengine metadata
        "matplotlib",
        "pyyaml",
        "Pillow",
        "pycocotools",
        "shapely",
        "scipy>=1.13,<2.0",
        "rich",
        "yapf",
        "six",
        "terminaltables",
        "tqdm",
    )
    _pip("--upgrade", "--no-deps", "mmengine>=0.10.5,<1.0.0")
    _pip(
        "--upgrade", "--no-cache-dir", "--only-binary=mmcv", "--no-deps",
        "mmcv==2.2.0", "-f", MMCV_INDEX,
    )
    _pip("--upgrade", "--no-deps", "mmdet==3.3.0")
    _patch_mmdet_mmcv_guard()

    # ── Repair disk numpy to >=2.0 AFTER all --no-deps installs. ─────────────
    # A prior broken Kaggle run may have left numpy 1.26.4 on disk (from an
    # older version of this script).  The running kernel always starts with
    # numpy 2.5.x from the Kaggle container image, so upgrading disk files
    # from 1.x → 2.x is safe: it brings the on-disk state back in sync with
    # the in-memory C extension (2.5.x), eliminating the mixed state that
    # causes `ImportError: _center` in fresh subprocesses.
    _pip("--upgrade", "numpy>=2.0.0,<3.0.0")

    # Verify in a fresh subprocess (clean import order, no mixed-state risk).
    try:
        print(f"  [deps] verified in a fresh interpreter: {_dependency_probe()}")
    except subprocess.CalledProcessError as error:
        output = getattr(error, "output", "") or "(no output)"
        print("  [deps] post-install probe FAILED — full output:\n" + output)
        raise RuntimeError(
            "OpenMMLab environment validation failed after installation. "
            "See probe output above."
        ) from error



# %% 2. Dataset discovery and fold-integrity validation
def _split_dirs(root: Path, split: str) -> tuple[Path, Path]:
    """Support Session-00 legacy and group-disjoint V2 directory layouts."""
    split_names = ("valid", "val", "validation") if split in {
        "valid", "val", "validation"
    } else (split,)
    for split_name in split_names:
        v2 = (root / "images" / split_name, root / "labels" / split_name)
        if all(path.is_dir() for path in v2):
            return v2
        legacy = (root / split_name / "images", root / split_name / "labels")
        if all(path.is_dir() for path in legacy):
            return legacy
    return root / split_names[0] / "images", root / split_names[0] / "labels"


def _valid_dataset_root(root: Path) -> bool:
    if not (root / "data.yaml").is_file():
        return False
    # Only train and valid are mandatory. test is used for group-isolation
    # checking when cv/ manifests are present, but is otherwise optional.
    return all(
        all(path.is_dir() for path in _split_dirs(root, split))
        for split in ("train", "valid")
    )


def _dataset_score(root: Path) -> tuple[int, int, int]:
    normalized_name = "".join(
        character for character in root.name.lower() if character.isalnum()
    )
    source_manifest = root / "manifest.csv"
    has_source_manifest = False
    if source_manifest.is_file():
        try:
            with source_manifest.open(newline="", encoding="utf-8") as handle:
                columns = set(next(csv.reader(handle), []))
            has_source_manifest = {"name", "split", "source", "augmented"} <= columns
        except OSError:
            pass
    return (
        int(normalized_name == "skindisnet2annotation"),
        int(has_source_manifest),
        int(all((root / "cv" / f"fold_{fold}" / "train.txt").is_file()
                and (root / "cv" / f"fold_{fold}" / "valid.txt").is_file()
                for fold in range(5))),
    )


def _find_dataset_root(hint: str) -> Path:
    if hint != "auto":
        explicit = Path(hint).expanduser().resolve()
        if _valid_dataset_root(explicit):
            print(f"  [data] explicit dataset: {explicit}")
            return explicit
        if explicit.is_dir():
            nested = sorted({
                data_yaml.parent.resolve()
                for data_yaml in explicit.rglob("data.yaml")
                if _valid_dataset_root(data_yaml.parent)
            })
            if len(nested) == 1:
                print(f"  [data] found dataset under explicit directory: {nested[0]}")
                return nested[0]
            if len(nested) > 1:
                choices = "\n".join(f"    - {path}" for path in nested)
                raise RuntimeError(
                    "Multiple complete datasets were found under --data-root. "
                    "Pass the dataset directory explicitly:\n" + choices
                )
        raise FileNotFoundError(
            f"--data-root is not a complete YOLO dataset: {explicit}"
        )

    candidates: set[Path] = set()
    search_roots = [Path("/kaggle/input")]
    if not search_roots[0].is_dir():
        search_roots.append(Path.cwd())
    for search_root in search_roots:
        if search_root.is_dir():
            for data_yaml in search_root.rglob("data.yaml"):
                root = data_yaml.parent.resolve()
                if _valid_dataset_root(root):
                    candidates.add(root)
    for candidate in (
        Path("/kaggle/working/SkinDisNet_2_Annotation"),
        Path("SkinDisNet_2_Annotation"),
        Path("../SkinDisNet_2_Annotation"),
        Path("/kaggle/working/Image_Annotation_rebuilt"),
        Path("Image_Annotation_rebuilt"),
        Path("../Image_Annotation_rebuilt"),
    ):
        if _valid_dataset_root(candidate):
            candidates.add(candidate.resolve())
    if not candidates:
        raise FileNotFoundError(
            "No complete dataset was found. Attach SkinDisNet_2_Annotation to "
            "Kaggle and keep DATA_ROOT='auto', or pass its mounted "
            "/kaggle/input/<dataset-slug> path explicitly."
        )

    ranked = sorted(candidates, key=lambda path: (_dataset_score(path), str(path)),
                    reverse=True)
    best_score = _dataset_score(ranked[0])
    tied = [path for path in ranked if _dataset_score(path) == best_score]
    if len(tied) > 1:
        choices = "\n".join(f"    - {path}" for path in tied)
        raise RuntimeError(
            "Multiple equally suitable datasets were found. Set "
            f"DATA_ROOT explicitly:\n{choices}"
        )
    print(f"  [data] selected dataset: {ranked[0]}")
    print(
        "  [data] preference features (SkinDisNet name, source manifest, CV)="
        f"{best_score}"
    )
    return ranked[0]


def _load_class_names(data_yaml: Path) -> list[str]:
    import yaml

    document = yaml.safe_load(data_yaml.read_text()) or {}
    names = document.get("names")
    if isinstance(names, dict):
        try:
            names = [
                str(names[key])
                for key in sorted(names, key=lambda value: int(value))
            ]
        except (TypeError, ValueError) as error:
            raise ValueError(
                f"{data_yaml} names mapping keys must be integer class IDs"
            ) from error
    names = [str(name).strip() for name in names] if isinstance(names, list) else []
    if not names or any(not name for name in names):
        raise ValueError(
            f"{data_yaml} must define a non-empty ordered 'names' list or mapping"
        )
    class_count = document.get("nc")
    if class_count is not None and int(class_count) != len(names):
        raise ValueError(
            f"{data_yaml} declares nc={class_count}, but contains {len(names)} names"
        )
    return names


def _gather_split(root: Path, split: str) -> list[dict]:
    images_dir, labels_dir = _split_dirs(root, split)
    items = []
    for image in sorted(images_dir.iterdir()):
        if image.suffix.lower() not in IMAGE_SUFFIXES:
            continue
        label = labels_dir / f"{image.stem}.txt"
        if not label.is_file():
            raise FileNotFoundError(f"Missing label for {image}: {label}")
        items.append({
            "image": image.resolve(),
            "label": label.resolve(),
            "key": str(image.resolve()),
            "split": {"val": "valid", "validation": "valid"}.get(split, split),
        })
    if not items:
        raise ValueError(f"No supported images found in {images_dir}")
    return items


def _item_classes(item: dict) -> set[int]:
    classes = set()
    for line in item["label"].read_text().splitlines():
        values = line.split()
        if values:
            classes.add(int(values[0]))
    return classes


def _manifest_lookup(pool: list[dict]) -> dict[str, dict]:
    lookup: dict[str, dict] = {}
    for item in pool:
        aliases = {item["key"], item["image"].name, item["image"].stem}
        for alias in aliases:
            if alias in lookup and lookup[alias]["key"] != item["key"]:
                raise RuntimeError(
                    f"Ambiguous image alias {alias}; fold manifests cannot be resolved"
                )
            lookup[alias] = item
    return lookup


def _lookup_manifest_item(value: str, lookup: dict[str, dict]) -> dict | None:
    candidate = Path(value)
    return lookup.get(value) or lookup.get(candidate.name) or lookup.get(candidate.stem)


def _read_fold_list(
    path: Path,
    lookup: dict[str, dict],
    excluded_lookup: dict[str, dict],
) -> tuple[list[dict], int, int]:
    resolved: dict[str, dict] = {}
    missing = 0
    excluded = 0
    for raw in path.read_text().splitlines():
        value = raw.strip()
        if not value or value.startswith("#"):
            continue
        item = _lookup_manifest_item(value, lookup)
        if item is None:
            if _lookup_manifest_item(value, excluded_lookup) is not None:
                excluded += 1
            else:
                missing += 1
        else:
            resolved[item["key"]] = item
    return [resolved[key] for key in sorted(resolved)], missing, excluded


def _verify_group_manifest(root: Path, folds: list[dict], test: list[dict]) -> dict:
    """Verify group isolation when the V2 metadata manifest is attached."""
    manifest = root / "metadata" / "manifest.csv"
    if not manifest.is_file():
        return {
            "group_manifest_present": False,
            "group_disjoint_verified": False,
            "reason": "metadata/manifest.csv is absent",
        }
    with manifest.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    by_basename = {}
    for row in rows:
        image_ref = row.get("image_file", "")
        if image_ref:
            by_basename[Path(image_ref).name] = row
    all_fold_items = [
        item for current in folds for item in current["valid"]
    ]
    required_items = all_fold_items + test
    missing_rows = [
        item["image"].name for item in required_items
        if item["image"].name not in by_basename
    ]
    missing_groups = [
        item["image"].name for item in required_items
        if item["image"].name in by_basename
        and not by_basename[item["image"].name].get("split_group_id", "")
    ]
    if missing_rows or missing_groups:
        raise RuntimeError(
            "V2 group manifest is incomplete: "
            f"missing_rows={len(missing_rows)}, missing_group_ids={len(missing_groups)}"
        )
    test_groups = {
        by_basename[item["image"].name].get("split_group_id", "")
        for item in test if item["image"].name in by_basename
    }
    test_groups.discard("")
    fold_report = []
    for current in folds:
        train_groups = {
            by_basename[item["image"].name].get("split_group_id", "")
            for item in current["train"] if item["image"].name in by_basename
        }
        valid_groups = {
            by_basename[item["image"].name].get("split_group_id", "")
            for item in current["valid"] if item["image"].name in by_basename
        }
        train_groups.discard("")
        valid_groups.discard("")
        train_valid_overlap = train_groups & valid_groups
        test_overlap = (train_groups | valid_groups) & test_groups
        if train_valid_overlap or test_overlap:
            raise RuntimeError(
                f"Group leakage found in fold {current['fold']}: "
                f"train/valid={len(train_valid_overlap)}, development/test={len(test_overlap)}"
            )
        fold_report.append({
            "fold": current["fold"],
            "train_valid_group_overlap": 0,
            "development_test_group_overlap": 0,
        })
    return {
        "group_manifest_present": True,
        "group_disjoint_verified": True,
        "fold_checks": fold_report,
    }


def _load_predefined_folds(
    root: Path, development: list[dict], test: list[dict]
) -> tuple[list[dict], dict] | None:
    fold_dirs = [root / "cv" / f"fold_{fold}" for fold in range(5)]
    if not all((folder / "train.txt").is_file()
               and (folder / "valid.txt").is_file() for folder in fold_dirs):
        return None
    lookup = _manifest_lookup(development)
    test_lookup = _manifest_lookup(test)
    development_keys = {item["key"] for item in development}
    validation_counts: Counter = Counter()
    folds = []
    stale = 0
    excluded_test = 0
    for fold, folder in enumerate(fold_dirs):
        train, train_missing, train_test = _read_fold_list(
            folder / "train.txt", lookup, test_lookup
        )
        valid, valid_missing, valid_test = _read_fold_list(
            folder / "valid.txt", lookup, test_lookup
        )
        stale += train_missing + valid_missing
        excluded_test += train_test + valid_test
        train_keys = {item["key"] for item in train}
        valid_keys = {item["key"] for item in valid}
        if train_keys & valid_keys:
            raise RuntimeError(f"Fold {fold} has train/validation image overlap")
        if train_keys | valid_keys != development_keys:
            absent = development_keys - (train_keys | valid_keys)
            raise RuntimeError(
                f"Fold {fold} does not cover {len(absent)} available development images"
            )
        validation_counts.update(valid_keys)
        folds.append({"fold": fold, "train": train, "valid": valid})
    bad = {key: count for key, count in validation_counts.items() if count != 1}
    if set(validation_counts) != development_keys or bad:
        raise RuntimeError(
            "Five-fold validation must cover every development image exactly once; "
            f"bad_count={len(bad)}"
        )
    group_check = _verify_group_manifest(root, folds, test)
    integrity = {
        "cv_mode": "predefined Session-00 five-fold manifests",
        "development_images": len(development),
        "untouched_test_images": len(test),
        "fold_val_sizes": {item["fold"]: len(item["valid"]) for item in folds},
        "test_manifest_references_excluded": excluded_test,
        "missing_manifest_references_filtered": stale,
        **group_check,
    }
    return folds, integrity


def _multi_hot(items: list[dict], num_classes: int) -> "np.ndarray":
    import numpy as np

    matrix = np.zeros((len(items), num_classes), dtype=np.int32)
    for index, item in enumerate(items):
        for class_id in _item_classes(item):
            if 0 <= class_id < num_classes:
                matrix[index, class_id] = 1
    return matrix


def _manifest_source_groups(root: Path, items: list[dict]) -> dict | None:
    """Attach dataset source-group IDs from either supported manifest format."""
    v2_manifest = root / "metadata" / "manifest.csv"
    source_manifest = root / "manifest.csv"
    if v2_manifest.is_file():
        with v2_manifest.open(newline="", encoding="utf-8") as handle:
            rows = list(csv.DictReader(handle))
        by_name = {
            Path(row.get("image_file", "")).name: row.get("split_group_id", "")
            for row in rows if row.get("image_file")
        }
        manifest_kind = "V2 split-group manifest"
        group_for_name = by_name
        name_for_item = lambda item: item["image"].name
    elif source_manifest.is_file():
        with source_manifest.open(newline="", encoding="utf-8") as handle:
            reader = csv.DictReader(handle)
            required = {"name", "split", "source", "augmented"}
            if not reader.fieldnames or not required <= set(reader.fieldnames):
                raise ValueError(
                    f"{source_manifest} must contain columns {sorted(required)}"
                )
            rows = list(reader)
        group_for_name = {}
        for row in rows:
            key = (
                row.get("split", "").strip().lower(),
                Path(row.get("name", "")).stem,
            )
            source = row.get("source", "").strip()
            if not key[1] or not source or key in group_for_name:
                raise ValueError("Incomplete or ambiguous dataset source manifest")
            group_for_name[key] = source
        manifest_kind = "dataset source manifest"
        name_for_item = lambda item: (
            item.get("split", "").lower(), item["image"].stem
        )
    else:
        return None

    missing = []
    for item in items:
        name = name_for_item(item)
        group = group_for_name.get(name)
        if not group:
            missing.append(str(item["image"]))
        else:
            item["group_id"] = group
    if missing:
        raise ValueError(
            f"{manifest_kind} is missing {len(missing)} image entries; "
            f"first missing: {missing[0]}"
        )
    return {
        "group_manifest_present": True,
        "source_manifest_present": True,
        "source_group_method": manifest_kind,
        "source_groups": len({item["group_id"] for item in items}),
    }


def _source_group_folds(
    development: list[dict], num_classes: int, seed: int
) -> tuple[list[dict], dict]:
    """Create balanced folds without separating manifest-linked variants."""
    import numpy as np

    groups: dict[str, list[int]] = {}
    for index, item in enumerate(development):
        groups.setdefault(item["group_id"], []).append(index)
    if len(groups) < 5:
        raise ValueError("Five-fold CV requires at least five independent source groups")

    labels = _multi_hot(development, num_classes)
    group_ids = sorted(groups)
    features = np.asarray([
        [int(labels[groups[group_id], class_id].sum())
         for class_id in range(num_classes)] + [len(groups[group_id])]
        for group_id in group_ids
    ], dtype=float)
    target = np.maximum(features.sum(axis=0) / 5.0, 1.0)
    best_assignment = None
    best_objective = float("inf")
    for trial in range(32):
        rng = random.Random(seed + trial)
        order = list(range(len(group_ids)))
        rng.shuffle(order)
        order.sort(key=lambda index: -float((features[index] / target).max()))
        counts = np.zeros((5, num_classes + 1), dtype=float)
        assignment = {}
        for index in order:
            costs = []
            for fold_id in range(5):
                projected = counts.copy()
                projected[fold_id] += features[index]
                costs.append(float(((projected / target) ** 2).sum()))
            minimum = min(costs)
            choices = [fold_id for fold_id, cost in enumerate(costs)
                       if abs(cost - minimum) < 1e-10]
            fold_id = rng.choice(choices)
            counts[fold_id] += features[index]
            assignment[group_ids[index]] = fold_id
        objective = int((counts[:, :num_classes] == 0).sum()) * 1e6 + float(
            (((counts - target) / target) ** 2).sum()
        )
        if objective < best_objective:
            best_objective = objective
            best_assignment = assignment
    assert best_assignment is not None

    folds = []
    expected_classes = set(range(num_classes))
    for fold_id in range(5):
        train = [
            item for item in development
            if best_assignment[item["group_id"]] != fold_id
        ]
        valid = [
            item for item in development
            if best_assignment[item["group_id"]] == fold_id
        ]
        if not train or not valid:
            raise ValueError(f"Fold {fold_id} is empty after source-group assignment")
        if {item["group_id"] for item in train} & {
            item["group_id"] for item in valid
        }:
            raise AssertionError("A source group crossed a fold boundary")
        for split_name, split_items in (("train", train), ("valid", valid)):
            present = set().union(*(_item_classes(item) for item in split_items))
            if present != expected_classes:
                raise ValueError(
                    f"Fold {fold_id} {split_name} split lacks classes "
                    f"{expected_classes - present}"
                )
        folds.append({"fold": fold_id, "train": train, "valid": valid})
    return folds, {
        "cv_mode": "manifest source-group-aware five-fold CV",
        "development_images": len(development),
        "group_manifest_present": True,
        "source_manifest_present": True,
        "group_disjoint_verified": True,
        "patient_disjoint_verified": False,
        "warning": (
            "Source groups are from the dataset manifest, not verified patient "
            "identities. Original images and augmented variants remain within one fold."
        ),
        "fold_val_sizes": {fold["fold"]: len(fold["valid"]) for fold in folds},
        "source_groups": len(groups),
    }


def _grouped_folds(
    development: list[dict], num_classes: int, seed: int
) -> tuple[list[dict], dict]:
    """Balance five folds while keeping all source-related images together."""
    import numpy as np

    grouped: dict[str, list[int]] = {}
    for index, item in enumerate(development):
        grouped.setdefault(item["group_id"], []).append(index)
    if len(grouped) < 5:
        raise ValueError("Five-fold CV requires at least five independent source groups")

    group_ids = sorted(grouped)
    labels = _multi_hot(development, num_classes)
    features = np.asarray([
        [int(labels[grouped[group_id], class_id].sum())
         for class_id in range(num_classes)]
        + [len(grouped[group_id])]
        for group_id in group_ids
    ], dtype=float)
    target = np.maximum(features.sum(axis=0) / 5.0, 1.0)
    best_assignment = None
    best_objective = float("inf")
    for trial in range(32):
        rng = random.Random(seed + trial)
        order = list(range(len(group_ids)))
        rng.shuffle(order)
        order.sort(key=lambda index: -float((features[index] / target).max()))
        counts = np.zeros((5, num_classes + 1), dtype=float)
        assignment = {}
        for index in order:
            costs = []
            for fold in range(5):
                projected = counts.copy()
                projected[fold] += features[index]
                costs.append(float(((projected / target) ** 2).sum()))
            low = min(costs)
            choices = [fold for fold, cost in enumerate(costs)
                       if abs(cost - low) < 1e-10]
            fold = rng.choice(choices)
            counts[fold] += features[index]
            assignment[group_ids[index]] = fold
        missing_classes = int((counts[:, :num_classes] == 0).sum())
        objective = missing_classes * 1e6 + float(
            (((counts - target) / target) ** 2).sum()
        )
        if objective < best_objective:
            best_objective = objective
            best_assignment = assignment
    assert best_assignment is not None

    folds = []
    for fold_id in range(5):
        valid = [
            item for item in development
            if best_assignment[item["group_id"]] == fold_id
        ]
        train = [
            item for item in development
            if best_assignment[item["group_id"]] != fold_id
        ]
        if not train or not valid:
            raise ValueError(f"Fold {fold_id} is empty after source-group assignment")
        train_groups = {item["group_id"] for item in train}
        valid_groups = {item["group_id"] for item in valid}
        if train_groups & valid_groups:
            raise AssertionError("Source group crossed train/validation boundary")
        all_classes = set(range(num_classes))
        for split_items, label in ((train, "train"), (valid, "valid")):
            present = set().union(*(_item_classes(item) for item in split_items))
            if present != all_classes:
                raise ValueError(f"Fold {fold_id} {label} split lacks classes {all_classes-present}")
        folds.append({"fold": fold_id, "train": train, "valid": valid})

    return folds, {
        "cv_mode": "source-group-aware five-fold CV",
        "development_images": len(development),
        "group_manifest_present": True,
        "group_disjoint_verified": True,
        "patient_disjoint_verified": False,
        "warning": (
            "Source groups come from the dataset manifest, not verified patient "
            "identities. Original images and augmented variants remain within one fold."
        ),
        "fold_val_sizes": {fold["fold"]: len(fold["valid"]) for fold in folds},
    }


def _fallback_folds(
    development: list[dict], num_classes: int, seed: int, root: Path | None = None
) -> tuple[list[dict], dict]:
    """Prefer manifest-backed source-group folds; otherwise use image-level CV.

    Pure-numpy implementation — no sklearn/scipy dependency.
    sklearn -> scipy -> numpy version-mismatch crashes the kernel (ImportError:
    _center from 'numpy._core.umath').
    """
    if root is not None:
        group_metadata = _manifest_source_groups(root, development)
        if group_metadata:
            return _source_group_folds(development, num_classes, seed)

    import numpy as np

    labels = _multi_hot(development, num_classes)
    dominant = labels.argmax(axis=1)

    # --- pure-numpy stratified k-fold -----------------------------------
    rng = np.random.RandomState(seed)
    n = len(development)
    n_splits = 5
    fold_of = np.full(n, -1, dtype=np.int32)
    classes = np.unique(dominant)
    for cls in classes:
        idx = np.where(dominant == cls)[0]
        rng.shuffle(idx)
        for k, chunk in enumerate(np.array_split(idx, n_splits)):
            for i in chunk:
                if fold_of[i] == -1:
                    fold_of[i] = k % n_splits
    # assign any still-unassigned (edge case: empty dominant class)
    unassigned = np.where(fold_of == -1)[0]
    if len(unassigned):
        for k, i in enumerate(unassigned):
            fold_of[i] = k % n_splits
    # -----------------------------------------------------------------------

    folds = []
    for fold in range(n_splits):
        folds.append({
            "fold": fold,
            "train": [item for index, item in enumerate(development)
                      if fold_of[index] != fold],
            "valid": [item for index, item in enumerate(development)
                      if fold_of[index] == fold],
        })
    return folds, {
        "cv_mode": "image-level stratified fallback",
        "development_images": len(development),
        "group_manifest_present": False,
        "group_disjoint_verified": False,
        "warning": (
            "No predefined cv manifests were found. These folds are image-disjoint "
            "but cannot prove patient/case/source-group isolation."
        ),
        "fold_val_sizes": {item["fold"]: len(item["valid"]) for item in folds},
    }


# %% 3. Model configuration and official checkpoint adaptation
def _detect_gpu_ids(device: str) -> list[str]:
    if device != "auto":
        identifiers = [part.strip() for part in device.split(",") if part.strip()]
        if device == "cpu" or not identifiers:
            raise RuntimeError("Session 24 requires a CUDA GPU")
        return identifiers
    count = int(subprocess.check_output(
        [sys.executable, "-c", "import torch; print(torch.cuda.device_count())"],
        text=True,
    ).strip())
    if count < 1:
        raise RuntimeError("No CUDA GPU detected. Enable a Kaggle GPU accelerator.")
    return [str(index) for index in range(min(2, count))]


def _write_training_config(
    destination: Path,
    train_json: Path,
    valid_json: Path,
    work_dir: Path,
    checkpoint: Path,
    names: list[str],
    gpu_count: int,
    max_iters: int,
    image_size: int,
    accumulation: int,
    seed: int,
) -> None:
    classes = repr(tuple(names))
    palette = repr([
        ((220 + 37 * index) % 256,
         (20 + 67 * index) % 256,
         (60 + 97 * index) % 256)
        for index in range(len(names))
    ])
    interval = max(20 if max_iters <= 100 else 500, max_iters // 4)
    warmup = min(500, max(10, max_iters // 10))
    effective_batch = gpu_count * accumulation  # batch_per_gpu is fixed at one
    learning_rate = 0.01 * effective_batch / 16.0
    milestone_1 = max(warmup + 1, int(max_iters * 0.80))
    milestone_2 = max(milestone_1 + 1, int(max_iters * 0.92))
    config = f'''# Auto-generated by SESSION 24.
_base_ = ['mmdet::condinst/condinst_r50_fpn_ms-poly-90k_coco_instance.py']

default_scope = 'mmdet'
metainfo = dict(classes={classes}, palette={palette})
image_size = ({image_size}, {image_size})

model = dict(
    backbone=dict(
        depth=101,
        frozen_stages=1,
        with_cp=False,
        init_cfg=dict(type='Pretrained', checkpoint='torchvision://resnet101'),
        # Genuine DCNv2 in all ResNet-101 bottlenecks of C3, C4 and C5.
        dcn=dict(type='DCNv2', deform_groups=1, fallback_on_stride=False),
        stage_with_dcn=(False, True, True, True),
    ),
    bbox_head=dict(num_classes={len(names)}),
    mask_head=dict(max_masks_to_train=150),
)

train_pipeline = [
    dict(type='LoadImageFromFile', to_float32=True),
    dict(type='LoadAnnotations', with_bbox=True, with_mask=True, poly2mask=True),
    dict(type='RandomFlip', prob=0.5),
    dict(type='RandomResize', scale=image_size, ratio_range=(0.75, 1.25),
         resize_type='Resize', keep_ratio=True),
    dict(type='RandomCrop', crop_size=image_size, crop_type='absolute',
         recompute_bbox=True, allow_negative_crop=True),
    dict(type='FilterAnnotations', min_gt_bbox_wh=(1.0, 1.0), by_mask=True),
    dict(type='Pad', size=image_size),
    dict(type='PackDetInputs'),
]
test_pipeline = [
    dict(type='LoadImageFromFile', to_float32=True),
    dict(type='Resize', scale=image_size, keep_ratio=True),
    dict(type='Pad', size=image_size),
    dict(type='PackDetInputs', meta_keys=(
        'img_id', 'img_path', 'ori_shape', 'img_shape', 'scale_factor')),
]

train_dataloader = dict(
    _delete_=True, batch_size=1, num_workers=2, persistent_workers=True,
    sampler=dict(type='InfiniteSampler', shuffle=True),
    dataset=dict(
        type='CocoDataset', data_root='', ann_file={str(train_json)!r},
        data_prefix=dict(img=''), metainfo=metainfo,
        filter_cfg=dict(filter_empty_gt=True, min_size=1),
        pipeline=train_pipeline,
    ),
)
val_dataloader = dict(
    _delete_=True, batch_size=1, num_workers=2, persistent_workers=True,
    drop_last=False, sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(
        type='CocoDataset', data_root='', ann_file={str(valid_json)!r},
        data_prefix=dict(img=''), metainfo=metainfo, test_mode=True,
        pipeline=test_pipeline,
    ),
)
test_dataloader = val_dataloader
val_evaluator = dict(
    _delete_=True, type='CocoMetric', ann_file={str(valid_json)!r},
    metric=['bbox', 'segm'], classwise=True,
)
test_evaluator = val_evaluator

optim_wrapper = dict(
    _delete_=True, type='AmpOptimWrapper', loss_scale='dynamic',
    accumulative_counts={accumulation},
    optimizer=dict(type='SGD', lr={learning_rate}, momentum=0.9,
                   weight_decay=0.0001),
    clip_grad=dict(max_norm=35.0, norm_type=2),
)
param_scheduler = [
    dict(type='LinearLR', start_factor=0.001, by_epoch=False,
         begin=0, end={warmup}),
    dict(type='MultiStepLR', begin=0, end={max_iters}, by_epoch=False,
         milestones=[{milestone_1}, {milestone_2}], gamma=0.1),
]
train_cfg = dict(
    _delete_=True, type='IterBasedTrainLoop', max_iters={max_iters},
    val_interval={interval},
)
val_cfg = dict(_delete_=True, type='ValLoop')
test_cfg = dict(_delete_=True, type='TestLoop')
default_hooks = dict(
    logger=dict(type='LoggerHook', interval=50),
    checkpoint=dict(
        type='CheckpointHook', by_epoch=False, interval={interval},
        save_last=True, save_best='coco/segm_mAP_50', rule='greater',
        max_keep_ckpts=2,
    ),
)
log_processor = dict(type='LogProcessor', window_size=50, by_epoch=False)
randomness = dict(seed={seed}, deterministic=False, diff_rank_seed=True)
env_cfg = dict(
    cudnn_benchmark=True,
    mp_cfg=dict(mp_start_method='fork', opencv_num_threads=0),
    dist_cfg=dict(backend='nccl'),
)
model_wrapper_cfg = dict(
    type='MMDistributedDataParallel',
    find_unused_parameters=True,
)
launcher = 'none'
work_dir = {str(work_dir)!r}
load_from = {str(checkpoint)!r}
resume = False
'''
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(config)


def _validate_model_build(config: Path) -> None:
    """Build in a fresh process and prove CondInst/R101/DCNv2 are genuine."""
    code = r'''
import sys
from mmengine.config import Config
from mmdet.registry import MODELS
from mmdet.utils import register_all_modules
from mmcv.ops import ModulatedDeformConv2dPack
register_all_modules(init_default_scope=True)
cfg = Config.fromfile(sys.argv[1])
model = MODELS.build(cfg.model)
assert model.__class__.__name__ == 'CondInst'
assert model.backbone.__class__.__name__ == 'ResNet'
assert model.backbone.depth == 101
assert tuple(model.backbone.stage_blocks) == (3, 4, 23, 3)
assert tuple(model.backbone.stage_with_dcn) == (False, True, True, True)
assert model.bbox_head.__class__.__name__ == 'CondInstBboxHead'
assert model.mask_head.__class__.__name__ == 'CondInstMaskHead'
assert model.bbox_head.num_params == 169
assert model.mask_head.num_params == 169
dcn = sum(isinstance(module, ModulatedDeformConv2dPack)
          for module in model.modules())
assert dcn == 30, f'expected 30 DCNv2 bottlenecks in C3-C5, found {dcn}'
print('architecture=CondInst; backbone=ResNet-101; '
      'DCNv2_C3-C5=30; dynamic_mask_parameters=169')
'''
    subprocess.check_call([sys.executable, "-c", code, str(config)])


def _prepare_condinst_checkpoint(work: Path, config: Path) -> tuple[Path, dict]:
    """Reuse compatible official CondInst heads while R101 initializes from ImageNet."""
    prepared = work / "pretrained" / "condinst_r50_heads_for_r101_dcn.pth"
    report_path = work / "pretrained" / "checkpoint_adaptation.json"
    if prepared.is_file() and prepared.stat().st_size > 1_000_000 \
            and report_path.is_file():
        print(f"  [weights] prepared checkpoint exists: {prepared}")
        return prepared, json.loads(report_path.read_text())

    raw = work / "pretrained" / "condinst_r50_coco_raw.pth"
    raw.parent.mkdir(parents=True, exist_ok=True)
    print("  [weights] downloading official MMDetection CondInst-R50 COCO weights ...")
    try:
        urlretrieve(CONDINST_R50_URL, raw)
    except Exception as error:
        raw.unlink(missing_ok=True)
        raise RuntimeError(
            "Official CondInst checkpoint download failed. Enable Kaggle Internet "
            "and rerun. No fallback model will be used."
        ) from error
    if raw.stat().st_size <= 1_000_000:
        raw.unlink(missing_ok=True)
        raise RuntimeError("Downloaded CondInst checkpoint is unexpectedly small")

    import torch
    from mmengine.config import Config
    from mmdet.registry import MODELS
    from mmdet.utils import register_all_modules

    register_all_modules(init_default_scope=True)
    cfg = Config.fromfile(str(config))
    model = MODELS.build(cfg.model)
    target = model.state_dict()
    checkpoint = torch.load(raw, map_location="cpu", weights_only=False)
    loaded = checkpoint.get("state_dict", checkpoint.get("model", checkpoint))
    source = {
        (key[7:] if key.startswith("module.") else key): value
        for key, value in loaded.items()
    }
    retained = {}
    rejected_backbone = []
    rejected_shape = []
    rejected_missing = []
    for key, value in source.items():
        if key.startswith("backbone."):
            rejected_backbone.append(key)
        elif key not in target:
            rejected_missing.append(key)
        elif tuple(value.shape) != tuple(target[key].shape):
            rejected_shape.append(key)
        else:
            retained[key] = value
    prefixes = {
        "neck": any(key.startswith("neck.") for key in retained),
        "bbox_head": any(key.startswith("bbox_head.") for key in retained),
        "mask_head": any(key.startswith("mask_head.") for key in retained),
        "dynamic_controller": any("bbox_head.controller" in key for key in retained),
    }
    if not all(prefixes.values()) or len(retained) < 50:
        raise RuntimeError(
            "Official checkpoint did not contain the expected compatible CondInst "
            f"neck/dynamic heads: {prefixes}; retained={len(retained)}"
        )
    adapted = {
        "state_dict": retained,
        "meta": {
            "source": CONDINST_R50_URL,
            "adaptation": (
                "Removed R50 backbone; retained only exact-shape FPN and CondInst "
                "head tensors. R101-DCNv2 uses torchvision ImageNet initialization."
            ),
        },
    }
    torch.save(adapted, prepared)
    raw.unlink(missing_ok=True)
    report = {
        "source_url": CONDINST_R50_URL,
        "retained_tensors": len(retained),
        "removed_r50_backbone_tensors": len(rejected_backbone),
        "removed_shape_mismatches": rejected_shape,
        "removed_unknown_tensors": len(rejected_missing),
        "required_prefixes_present": prefixes,
    }
    report_path.write_text(json.dumps(report, indent=2))
    print(
        f"  [weights] retained={len(retained)} exact-shape FPN/head tensors; "
        f"removed_backbone={len(rejected_backbone)}; "
        f"shape_mismatch={len(rejected_shape)}"
    )
    return prepared, report


def _write_prediction_config(
    train_config: Path, destination: Path, valid_json: Path, prefix: Path
) -> None:
    override = f'''

# Strict out-of-fold prediction override generated by SESSION 24.
test_dataloader = dict(
    _delete_=True, batch_size=1, num_workers=2, persistent_workers=True,
    drop_last=False, sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(
        type='CocoDataset', data_root='', ann_file={str(valid_json)!r},
        data_prefix=dict(img=''), metainfo=metainfo, test_mode=True,
        pipeline=test_pipeline,
    ),
)
test_evaluator = dict(
    _delete_=True, type='CocoMetric', ann_file={str(valid_json)!r},
    metric=['bbox', 'segm'], format_only=True,
    outfile_prefix={str(prefix)!r},
)
load_from = None
resume = False
'''
    destination.write_text(train_config.read_text() + override)


# %% 4. Resume-safe completion markers
def _completed_checkpoint(work_dir: Path, config: Path) -> Path | None:
    marker = work_dir / "SESSION_24_TRAINING_COMPLETE.json"
    if not marker.is_file():
        return None
    try:
        record = json.loads(marker.read_text())
        digest = hashlib.sha256(config.read_bytes()).hexdigest()
        checkpoint = Path(record["checkpoint"])
        if record.get("config_sha256") == digest and checkpoint.is_file():
            return checkpoint
    except (KeyError, OSError, ValueError, json.JSONDecodeError):
        pass
    return None


def _mark_training_complete(work_dir: Path, config: Path, checkpoint: Path) -> None:
    marker = work_dir / "SESSION_24_TRAINING_COMPLETE.json"
    marker.write_text(json.dumps({
        "checkpoint": str(checkpoint),
        "config_sha256": hashlib.sha256(config.read_bytes()).hexdigest(),
        "completed_utc": datetime.now(timezone.utc).isoformat(),
    }, indent=2))


def _validate_training_resume(
    work_dir: Path,
    config: Path,
    train_json: Path,
    valid_json: Path,
    resume: bool,
) -> None:
    """Prevent resuming optimizer/scheduler state under a different run setup."""
    latest = work_dir / "latest.pth"
    run_config = work_dir / "SESSION_24_RUN_CONFIG.json"
    signature = {
        "config_sha256": hashlib.sha256(config.read_bytes()).hexdigest(),
        "train_sha256": hashlib.sha256(train_json.read_bytes()).hexdigest(),
        "valid_sha256": hashlib.sha256(valid_json.read_bytes()).hexdigest(),
    }

    if resume:
        if not latest.is_file():
            raise FileNotFoundError(
                f"--resume requires an interrupted-run checkpoint: {latest}"
            )
        if not run_config.is_file():
            raise RuntimeError(
                f"Cannot safely resume {work_dir}: no run signature exists. "
                "Use a new --work-dir and train with the updated settings."
            )
        saved = json.loads(run_config.read_text())
        if saved != signature:
            raise RuntimeError(
                f"Cannot resume {work_dir}: config or fold annotations changed. "
                "Use a new --work-dir; the optimizer and LR schedule cannot be "
                "safely resumed across this change."
            )
        return

    if latest.is_file() and not run_config.is_file():
        raise RuntimeError(
            f"An untracked checkpoint already exists in {work_dir}. Use a new "
            "--work-dir to start the longer schedule without mixing checkpoints."
        )
    if latest.is_file():
        saved = json.loads(run_config.read_text())
        if saved != signature:
            raise RuntimeError(
                f"Checkpoint in {work_dir} belongs to different settings. "
                "Use a new --work-dir rather than mixing training schedules."
            )
    run_config.write_text(json.dumps(signature, indent=2))


# %% 5. End-to-end Session 24 run
def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(
        description="SESSION 24: CondInst ResNet-101 DCNv2 five-fold OOF CV"
    )
    parser.add_argument("--data-root", default=os.environ.get("DATA_ROOT", "auto"))
    parser.add_argument("--work-dir", default=None)
    parser.add_argument("--device", default="auto", help="auto, 0, or 0,1")
    parser.add_argument("--folds", default="0,1,2,3,4")
    parser.add_argument(
        "--max-iters", type=int, default=6000,
        help=(
            "Training iterations per fold (default 6000; 30000 across five "
            "folds). Runtime depends on Kaggle GPU speed."
        ),
    )
    parser.add_argument("--image-size", type=int, default=640)
    parser.add_argument("--accumulation", type=int, default=2)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--quick", action="store_true")
    parser.add_argument("--skip-existing", action="store_true", default=True)
    parser.add_argument("--no-skip-existing", dest="skip_existing", action="store_false")
    parser.add_argument("--resume", action="store_true")
    parser.add_argument("--skip-train", action="store_true")
    args, _ = parser.parse_known_args(argv)

    if args.max_iters < 1:
        raise ValueError("--max-iters must be positive")
    if args.image_size < 384 or args.image_size % 32:
        raise ValueError("--image-size must be >=384 and divisible by 32")
    if args.accumulation < 1:
        raise ValueError("--accumulation must be positive")

    relaunch_code = _relaunch_python312(args)
    if relaunch_code is not None:
        return relaunch_code

    multiprocessing.freeze_support()
    started = time.time()
    print("=" * 76)
    print("SESSION 24 — genuine CondInst R101-DCNv2 + five-fold OOF CV")
    print("=" * 76)
    _ensure_dependencies()
    gpu_ids = _detect_gpu_ids(args.device)
    selected_folds = _parse_folds(args.folds)
    print(f"  [gpu] training devices={gpu_ids}; folds={selected_folds}")

    root = _find_dataset_root(args.data_root)
    names = _load_class_names(root / "data.yaml")
    train_items = _gather_split(root, "train")
    valid_items = _gather_split(root, "valid")
    # The test split is optional for older exports; when present, manifest
    # source groups are checked against it but it never enters CV training.
    try:
        test_items = _gather_split(root, "test")
    except (FileNotFoundError, ValueError):
        test_items = []
    development = train_items + valid_items
    if len({item["key"] for item in development}) != len(development):
        raise RuntimeError("Duplicate image paths exist in the development pool")
    print(
        f"  [data] development={len(development)} "
        f"({len(train_items)} train + {len(valid_items)} valid); "
        f"untouched_test={len(test_items)}"
    )
    dataset_evaluation_note = None
    dataset_summary_path = root / "summary.json"
    if dataset_summary_path.is_file():
        dataset_summary = json.loads(
            dataset_summary_path.read_text(encoding="utf-8")
        )
        dataset_evaluation_note = dataset_summary.get("evaluation_note")
        if dataset_evaluation_note:
            print(f"  [data] evaluation note: {dataset_evaluation_note}")
    group_metadata = _manifest_source_groups(root, development + test_items)
    if group_metadata:
        development_groups = {item["group_id"] for item in development}
        test_groups = {item["group_id"] for item in test_items}
        if development_groups & test_groups:
            raise RuntimeError(
                "Dataset source manifest places related development and test "
                "images in different splits."
            )
    predefined = _load_predefined_folds(root, development, test_items)
    if predefined is None:
        fold_data, integrity = _fallback_folds(
            development, len(names), args.seed, root
        )
        if integrity["group_disjoint_verified"]:
            print("  [integrity] using source-manifest group-aware folds")
        else:
            print("  [integrity] WARNING: using image-level fallback folds")
    else:
        fold_data, integrity = predefined
        print(
            "  [integrity] predefined five-fold manifests validated; "
            f"group_disjoint_verified={integrity['group_disjoint_verified']}"
        )
    if group_metadata:
        for fold in fold_data:
            train_groups = {item["group_id"] for item in fold["train"]}
            valid_groups = {item["group_id"] for item in fold["valid"]}
            if train_groups & valid_groups:
                raise RuntimeError(
                    f"Fold {fold['fold']} separates related manifest source images."
                )
        integrity.update({
            "source_manifest_present": True,
            "source_group_method": group_metadata["source_group_method"],
            "source_groups": group_metadata["source_groups"],
            "group_disjoint_verified": True,
            "development_test_group_overlap": 0,
        })

    default_work = Path("/kaggle/working") / (
        f"{SESSION_NAME}_quick" if args.quick else SESSION_NAME
    )
    work = Path(args.work_dir).expanduser().resolve() if args.work_dir else default_work
    work.mkdir(parents=True, exist_ok=True)
    coco_dir = work / "coco"
    configs_dir = work / "configs"
    predictions_dir = work / "oof_predictions"
    for directory in (coco_dir, configs_dir, predictions_dir):
        directory.mkdir(parents=True, exist_ok=True)

    if args.quick:
        selected_folds = [selected_folds[0]]
        args.max_iters = min(args.max_iters, 20)
        print("  [quick] one fold, <=48 train / <=24 validation images, 20 iterations")
    selected_data = []
    for fold in selected_folds:
        current = dict(fold_data[fold])
        if args.quick:
            current["train"] = _balanced_subset(
                current["train"], 48, args.seed, len(names)
            )
            current["valid"] = _balanced_subset(
                current["valid"], 24, args.seed + 1, len(names)
            )
        selected_data.append(current)

    all_items_by_key = {
        item["key"]: item
        for current in selected_data
        for item in current["train"] + current["valid"]
    }
    image_ids = {
        key: index + 1 for index, key in enumerate(sorted(all_items_by_key))
    }
    oof_items_by_key = {
        item["key"]: item for current in selected_data for item in current["valid"]
    }
    oof_items = [oof_items_by_key[key] for key in sorted(oof_items_by_key)]
    oof_gt = coco_dir / "selected_oof_ground_truth.json"
    oof_conversion = _yolo_to_coco(oof_items, names, oof_gt, image_ids)

    # Write a representative config before adapting the official checkpoint.
    bootstrap_config = configs_dir / "architecture_preflight.py"
    prepared_path = work / "pretrained" / "condinst_r50_heads_for_r101_dcn.pth"
    _write_training_config(
        bootstrap_config, oof_gt, oof_gt, work / "preflight", prepared_path,
        names, len(gpu_ids), args.max_iters, args.image_size,
        args.accumulation, args.seed,
    )
    compile(bootstrap_config.read_text(), str(bootstrap_config), "exec")
    print("  [preflight] building model and verifying CondInst/R101/DCNv2 ...")
    _validate_model_build(bootstrap_config)
    pretrained, adaptation_report = _prepare_condinst_checkpoint(
        work, bootstrap_config
    )

    summary = {
        "session": "SESSION_24",
        "model": "CondInst with ResNet-101 and DCNv2 in C3-C5",
        "method": (
            "five-fold cross-validation with strict out-of-fold evaluation; "
            "CondInst per-instance dynamic convolution masks"
        ),
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_root": str(root),
        "dataset_evaluation_note": dataset_evaluation_note,
        "classes": names,
        "selected_folds": selected_folds,
        "dataset_integrity": integrity,
        "oof_conversion": oof_conversion,
        "checkpoint_adaptation": adaptation_report,
        "configuration": {
            "max_iters_per_fold": args.max_iters,
            "image_size": args.image_size,
            "gpus": gpu_ids,
            "batch_per_gpu": 1,
            "gradient_accumulation": args.accumulation,
            "quick": args.quick,
            "backbone": "ResNet-101",
            "dcn_type": "DCNv2",
            "dcn_stages": ["C3", "C4", "C5"],
            "verified_dcn_layers": 30,
            "dynamic_mask_parameters_per_instance": 169,
        },
        "fold_results": [],
        "oof": {},
        "limitations": [
            "The 0.73-0.78 mAP50 range in model_strategy_analysis.md is a "
            "planning estimate, not a measured result or guarantee.",
            "The untouched test split is not evaluated during CV; use it once "
            "after model selection for a final locked estimate.",
            "Source-group separation is based on manifest groups, not verified "
            "patient identities.",
            *(
                [dataset_evaluation_note]
                if dataset_evaluation_note else []
            ),
            "MMDetection 3.3 with the Python-3.12 MMCV 2.2 wheel uses a narrowly "
            "scoped version-guard compatibility adjustment.",
        ],
    }
    summary_path = work / "SESSION_24_results.json"

    def save_summary() -> None:
        summary_path.write_text(json.dumps(summary, indent=2, default=str))

    prediction_files = []
    for current in selected_data:
        fold = current["fold"]
        print("\n" + "-" * 76)
        print(f"FOLD {fold}: train={len(current['train'])}, valid={len(current['valid'])}")
        print("-" * 76)
        fold_coco = coco_dir / f"fold_{fold}"
        fold_config_dir = configs_dir / f"fold_{fold}"
        fold_work = work / "folds" / f"fold_{fold}" / f"iter_{args.max_iters}"
        for directory in (fold_coco, fold_config_dir, fold_work):
            directory.mkdir(parents=True, exist_ok=True)
        train_json = fold_coco / "train.json"
        valid_json = fold_coco / "valid.json"
        train_report = _yolo_to_coco(current["train"], names, train_json, image_ids)
        valid_report = _yolo_to_coco(current["valid"], names, valid_json, image_ids)
        config = fold_config_dir / "condinst_r101_dcn.py"
        _write_training_config(
            config, train_json, valid_json, fold_work, pretrained, names,
            len(gpu_ids), args.max_iters, args.image_size, args.accumulation,
            args.seed + fold,
        )
        compile(config.read_text(), str(config), "exec")

        checkpoint = _best_checkpoint(fold_work)
        completed_checkpoint = _completed_checkpoint(fold_work, config)
        if args.skip_train:
            if checkpoint is None:
                raise FileNotFoundError(
                    f"--skip-train was set but fold {fold} has no checkpoint"
                )
        elif args.skip_existing and completed_checkpoint is not None:
            checkpoint = completed_checkpoint
            print(f"  [train] reusing completed fold checkpoint: {checkpoint}")
        else:
            _validate_training_resume(
                fold_work, config, train_json, valid_json, args.resume
            )
            _run_training(config, fold_work, gpu_ids, args.resume)
            checkpoint = _best_checkpoint(fold_work)
            if checkpoint is None:
                raise RuntimeError(f"Fold {fold} finished without a checkpoint")
            _mark_training_complete(fold_work, config, checkpoint)

        prediction_prefix = predictions_dir / f"fold_{fold}_valid"
        prediction_config = fold_config_dir / "predict_valid.py"
        _write_prediction_config(
            config, prediction_config, valid_json, prediction_prefix
        )
        print(f"  [oof] predicting fold {fold} validation with {checkpoint.name}")
        _run_prediction(prediction_config, checkpoint, fold_work, gpu_ids[0])
        prediction_json = Path(f"{prediction_prefix}.segm.json")
        if not prediction_json.is_file():
            raise FileNotFoundError(f"MMDetection did not create {prediction_json}")
        metrics = _coco_evaluate(valid_json, prediction_json, names)
        prediction_files.append(prediction_json)
        summary["fold_results"].append({
            "fold": fold,
            "checkpoint": str(checkpoint),
            "train_conversion": train_report,
            "valid_conversion": valid_report,
            "metrics": metrics,
        })
        save_summary()

    oof_predictions = predictions_dir / "SESSION_24_oof.segm.json"
    detection_count = _merge_oof_predictions(prediction_files, oof_predictions)
    oof_metrics = _coco_evaluate(oof_gt, oof_predictions, names)
    fold_scores = [
        result["metrics"]["segm_mAP50"] for result in summary["fold_results"]
    ]
    summary["oof"] = {
        "evaluated_images": len(oof_items),
        "detections": detection_count,
        "metrics": oof_metrics,
        "fold_mean_segm_mAP50": float(statistics.mean(fold_scores)),
        "fold_std_segm_mAP50": (
            float(statistics.pstdev(fold_scores)) if len(fold_scores) > 1 else 0.0
        ),
        "complete_five_fold_oof": selected_folds == [0, 1, 2, 3, 4]
        and not args.quick,
    }
    summary["elapsed_hours"] = (time.time() - started) / 3600.0
    save_summary()

    archive = Path("/kaggle/working/SESSION_24_results.zip")
    if not Path("/kaggle/working").is_dir():
        archive = work / "SESSION_24_results.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as output_zip:
        output_zip.write(summary_path, arcname=summary_path.name)
        output_zip.write(oof_gt, arcname=oof_gt.name)
        output_zip.write(oof_predictions, arcname=oof_predictions.name)
        adaptation_path = work / "pretrained" / "checkpoint_adaptation.json"
        output_zip.write(adaptation_path, arcname=adaptation_path.name)
        for current, prediction in zip(selected_data, prediction_files):
            fold = current["fold"]
            config = configs_dir / f"fold_{fold}" / "condinst_r101_dcn.py"
            output_zip.write(config, arcname=f"fold_{fold}/{config.name}")
            output_zip.write(prediction, arcname=f"fold_{fold}/{prediction.name}")

    print("\n" + "=" * 76)
    print("SESSION 24 COMPLETE")
    print("=" * 76)
    print(
        f"OOF segm mAP50={oof_metrics['segm_mAP50']:.4f}; "
        f"mAP50-95={oof_metrics['segm_mAP50_95']:.4f}; "
        f"bbox mAP50={oof_metrics['bbox_mAP50']:.4f}"
    )
    print("Per-class OOF segmentation mAP50:")
    for name, value in oof_metrics["per_class_mAP50"].items():
        print(f"  {name}: {value:.4f}")
    print(f"Results JSON: {summary_path}")
    print(f"Results ZIP : {archive}")
    print(f"Checkpoints : {work / 'folds'}  (download separately)")
    return 0


if __name__ == "__main__":
    multiprocessing.freeze_support()
    exit_code = main()
    try:
        sys.exit(exit_code)
    except SystemExit as error:
        if error.code:
            raise


  [install] uv
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 90.8 MB/s eta 0:00:00


 Downloaded cpython-3.12.15-linux-x86_64-gnu (download)
Installed Python 3.12.15 in 1.05s
 + cpython-3.12.15-linux-x86_64-gnu (python3.12)
Using CPython 3.12.15
Creating virtual environment with seed packages at: session24_condinst_py312
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.


  [python] restarting under Python 3.12: /kaggle/working/session24_condinst_py312/bin/python


 + pip==26.2.1
Activate with: source session24_condinst_py312/bin/activate


SESSION 24 — genuine CondInst R101-DCNv2 + five-fold OOF CV
  [deps] current environment probe failed:
Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'torch'
  [deps] installing pinned Kaggle packages without openmim ...
  [deps] isolated Python 3.12 environment has no PyTorch; installing torch 2.4.0/torchvision 0.19.0 with CUDA 12.1 ...
  [install] --upgrade --force-reinstall --no-cache-dir torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
  [install] --upgrade addict termcolor packaging opencv-python==4.10.0.84 matplotlib pyyaml Pillow pycocotools shapely scipy>=1.13,<2.0 rich yapf six terminaltables tqdm
  [install] --upgrade --no-deps mmengine>=0.10.5,<1.0.0
  [install] --upgrade --no-cache-dir --only-binary=mmcv --no-deps mmcv==2.2.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html
  [install] --upgrade --no-deps mmdet==3.3.0
  [deps] adjusted MMDetection's MMCV g

/kaggle/working/session24_condinst_py312/lib/python3.12/site-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


... (81 outputs left out)

/kaggle/working/session24_condinst_py312/lib/python3.12/site-packages/mmengine/runner/checkpoint.py:347: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.loa

10/10 20:34:19 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.16.conv2 is upgraded to version 2.
10/10 20:34:19 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.17.conv2 is upgraded to version 2.
10/10 20:34:19 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.18.conv2 is upgraded to version 2.
10/10 20:34:19 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.19.conv2 is upgraded to version 2.
10/10 20:34:19 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.20.conv2 is upgraded to version 2.
10/10 20:34:19 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.21.conv2 is upgraded to version 2.
10/10 20:34:19 - mmengine - INFO - ModulatedDeformConvPack backbone.layer3.22.conv2 is upgraded to version 2.
10/10 20:34:19 - mmengine - INFO - ModulatedDeformConvPack backbone.layer4.0.conv2 is upgraded to version 2.
10/10 20:34:19 - mmengine - INFO - ModulatedDeformConvPack backbone.layer4.1.conv2 is upgraded to version 2.
10/10 20:34:

/kaggle/working/session24_condinst_py312/lib/python3.12/site-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


10/10 20:34:29 - mmengine - INFO - Iter(test) [  50/1000]    eta: 0:03:12  time: 0.2027  data_time: 0.0040  memory: 8995  
10/10 20:34:34 - mmengine - INFO - Iter(test) [ 100/1000]    eta: 0:02:17  time: 0.1037  data_time: 0.0024  memory: 354  
10/10 20:34:39 - mmengine - INFO - Iter(test) [ 150/1000]    eta: 0:01:56  time: 0.1032  data_time: 0.0022  memory: 369  
10/10 20:34:45 - mmengine - INFO - Iter(test) [ 200/1000]    eta: 0:01:44  time: 0.1122  data_time: 0.0022  memory: 409  
10/10 20:34:51 - mmengine - INFO - Iter(test) [ 250/1000]    eta: 0:01:34  time: 0.1096  data_time: 0.0023  memory: 394  
10/10 20:34:56 - mmengine - INFO - Iter(test) [ 300/1000]    eta: 0:01:27  time: 0.1150  data_time: 0.0023  memory: 463  
10/10 20:35:02 - mmengine - INFO - Iter(test) [ 350/1000]    eta: 0:01:19  time: 0.1069  data_time: 0.0022  memory: 369  
10/10 20:35:07 - mmengine - INFO - Iter(test) [ 400/1000]    eta: 0:01:12  time: 0.1083  data_time: 0.0023  memory: 445  
10/10 20:35:13 - mmengi